# Automotive C Code Analyzer with MISRA-C Compliance and Unit Test Generation

This notebook demonstrates a multi-agent system for automotive C code analysis using Strands agents deployed on Amazon Bedrock AgentCore Runtime with inbound authentication.

## Overview

The system implements a graph multi-agent pattern with 2 steps:
1. **MISRA-C Compliance Analysis**: Analyzes C code for MISRA-C compliance violations
2. **Unit Test Generation**: If no severe violations are found, generates comprehensive unit tests

### Architecture

- **Agent 1**: MISRA-C Compliance Analyzer
- **Agent 2**: Unit Test Generator
- **Authentication**: Amazon Cognito with JWT tokens
- **Runtime**: Amazon Bedrock AgentCore Runtime
- **Model**: Anthropic Claude Sonnet 3.7

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Multi-agent automotive analysis                                                  |
| Agent type          | Graph pattern with 2 agents                                                     |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Sonnet 3.7                                                     |
| Tutorial components | MISRA-C analysis, Unit test generation, AgentCore Runtime                       |
| Tutorial vertical   | Automotive                                                                       |
| Example complexity  | Intermediate                                                                     |
| Inbound Auth        | Cognito                                                                          |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                    |

## Prerequisites

To execute this tutorial you will need:
* Python 3.13+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker running

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Setting up Amazon Cognito for Authentication

Let's provision a Cognito User Pool with an App client and one test user for automotive applications.

In [ ]:
import sys
import os
import time

# Import our utility functions
from utils import setup_cognito_user_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool for automotive applications...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")

## Preparing the Automotive C Code Analyzer Agent Graph

Let's examine our multi-agent graph system that implements MISRA-C compliance checking and conditional unit test generation using the Strands GraphBuilder pattern.

### Graph Architecture

Our system uses a **conditional sequential graph** pattern:

```
┌─────────────────────┐    Condition: No HIGH    ┌─────────────────────┐
│   MISRA Analyzer    │────severity violations───▶│  Unit Test Generator │
│   (Entry Point)     │                          │  (Conditional Node)  │
└─────────────────────┘                          └─────────────────────┘
```

- **Node 1**: MISRA-C Compliance Analyzer analyzes code for violations
- **Conditional Edge**: Only proceeds to Node 2 if no severe violations found
- **Node 2**: Unit Test Generator creates comprehensive tests (only if condition met)

This ensures that unit tests are only generated for code that meets automotive safety standards.

## Deploying the Agent to AgentCore Runtime

Now we'll configure and deploy our automotive C code analyzer to AgentCore Runtime with inbound authentication.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

discovery_url = cognito_config.get("discovery_url")
client_id = cognito_config.get("client_id")

print(f"Discovery URL: {discovery_url}")
print(f"Client ID: {client_id}")

### Configure AgentCore Runtime Deployment

In [ ]:
agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="c_code_analyzer.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="automotive_c_analyzer_inbound_auth",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id]
        }
    }
)
print("Configuration response:", response)

### Review the AgentCore Configuration

In [ ]:
!cat .bedrock_agentcore.yaml

### Launch Agent to AgentCore Runtime

In [ ]:
print("Launching automotive C code analyzer to AgentCore Runtime...")
launch_result = agentcore_runtime.launch()
print("Launch result:", launch_result)

### Check AgentCore Runtime Status

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

print(f"Initial status: {status}")
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"Status: {status}")

print(f"Final status: {status}")

## Testing the Automotive C Code Analyzer

Let's test our multi-agent system with sample C code that has MISRA-C violations.

### Test Case 1: C Code with Severe MISRA Violations

In [ ]:
# Load sample C code with severe MISRA violations from file
with open('sample_automotive_code/sample_with_severe_violations.c', 'r') as f:
    c_code_with_violations = f.read()

print("Sample C code with severe violations:")
print("=" * 50)
print(c_code_with_violations[:500] + "...")
print("=" * 50)

# Get fresh bearer token
bearer_token = reauthenticate_user(cognito_config.get("client_id"))

# Test with severe violations
print("\nTesting C code with severe MISRA violations...")
invoke_response = agentcore_runtime.invoke(
    {"c_code": c_code_with_violations}, 
    bearer_token=bearer_token
)
print("Response:", invoke_response)
print("Bearer token:", bearer_token)

### Test Case 2: C Code with Minor MISRA Violations

In [ ]:
# Load sample C code with minor MISRA violations from file
with open('sample_automotive_code/sample_with_minor_violations.c', 'r') as f:
    c_code_minor_violations = f.read()

print("Sample C code with minor violations:")
print("=" * 50)
print(c_code_minor_violations[:500] + "...")
print("=" * 50)

# Test with minor violations (should generate unit tests)
print("\nTesting C code with minor MISRA violations...")
invoke_response = agentcore_runtime.invoke(
    {"c_code": c_code_minor_violations}, 
    bearer_token=bearer_token
)
print("Response:", invoke_response)

### Test Case 3: MISRA-Compliant C Code

In [ ]:
# Load MISRA-compliant sample C code from file
with open('sample_automotive_code/sample_misra_compliant.c', 'r') as f:
    c_code_compliant = f.read()

print("Sample MISRA-compliant C code:")
print("=" * 50)
print(c_code_compliant[:500] + "...")
print("=" * 50)

# Test with compliant code (should generate unit tests)
print("\nTesting MISRA-compliant C code...")
invoke_response = agentcore_runtime.invoke(
    {"c_code": c_code_compliant}, 
    bearer_token=bearer_token
)
print("Response:", invoke_response)

## Testing Without Authorization (Should Fail)

In [ ]:
# This should fail with AccessDeniedException
try:
    print("Testing without authorization (should fail)...")
    invoke_response = agentcore_runtime.invoke({"c_code": c_code_compliant})
    print("Unexpected success:", invoke_response)
except Exception as e:
    print(f"Expected error: {e}")

## Cleanup (Optional)

Clean up the AgentCore Runtime and ECR repository when done testing.

In [ ]:
import boto3

# Uncomment to clean up resources
# agentcore_control_client = boto3.client(
#     'bedrock-agentcore-control',
#     region_name=region
# )
# ecr_client = boto3.client(
#     'ecr',
#     region_name=region
# )

# runtime_delete_response = agentcore_control_client.delete_agent_runtime(
#     agentRuntimeId=launch_result.agent_id,
# )

# response = ecr_client.delete_repository(
#     repositoryName=launch_result.ecr_uri.split('/')[1],
#     force=True
# )

print("Cleanup code is commented out. Uncomment to clean up resources.")

## Summary

This notebook demonstrated:

1. **Multi-Agent Graph Architecture**: Implemented using Strands GraphBuilder with conditional execution
   - MISRA-C Compliance Analyzer (Entry Node)
   - Unit Test Generator (Conditional Node)
   - Conditional edge based on violation severity

2. **Conditional Workflow**: Unit tests are only generated if no severe MISRA violations are found
   - Graph execution metrics and node tracking
   - Automatic workflow control based on analysis results

3. **Automotive Focus**: Specialized for automotive C code analysis with safety standards
   - MISRA-C:2012 compliance checking
   - Safety-critical code validation

4. **Security**: Implemented inbound authentication using Amazon Cognito
   - JWT token validation
   - Secure multi-agent execution

5. **Scalability**: Deployed on Amazon Bedrock AgentCore Runtime for production use
   - Graph-based agent orchestration
   - Performance monitoring and metrics

The system provides a comprehensive solution for automotive software development, using advanced multi-agent graph patterns to ensure code quality and safety compliance before proceeding with test generation.